In [1]:
from pathlib import Path
import html
import json
import re
import unicodedata

import numpy as np
import pandas as pd

from IPython.display import display


# =========================
# Пути проекта
# =========================

BASE_DIR = Path(r"D:\sber")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

NEWS_PATH = DATA_DIR / "news.csv"
CLIENTS_PATH = DATA_DIR / "clients.json"
INTERNAL_NOTES_PATH = DATA_DIR / "internal_notes.json"
PRODUCTS_PATH = DATA_DIR / "products.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Проверяем существование файлов
required_files = [
    NEWS_PATH,
    CLIENTS_PATH,
    INTERNAL_NOTES_PATH,
    PRODUCTS_PATH,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    missing_text = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f"Не найдены следующие файлы:\n{missing_text}"
    )

print("Все файлы найдены.")

Все файлы найдены.


In [2]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def read_news_csv(path: Path) -> tuple[pd.DataFrame, str, str]:
    """
    Загружает news.csv, перебирая распространённые
    кодировки и разделители.
    """

    required_columns = {"title", "text", "source"}

    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1251",
    ]

    separators = [
        ",",
        ";",
        "\t",
    ]

    errors = []

    for encoding in encodings:
        for separator in separators:
            try:
                dataframe = pd.read_csv(
                    path,
                    encoding=encoding,
                    sep=separator,
                )

                # Убираем случайные пробелы в названиях колонок
                dataframe.columns = [
                    str(column).strip()
                    for column in dataframe.columns
                ]

                if required_columns.issubset(dataframe.columns):
                    return dataframe, encoding, separator

            except Exception as error:
                errors.append(
                    {
                        "encoding": encoding,
                        "separator": repr(separator),
                        "error": str(error),
                    }
                )

    raise ValueError(
        "Не удалось корректно прочитать news.csv. "
        "Ожидались колонки title, text и source."
    )


clients = load_json(CLIENTS_PATH)
internal_notes = load_json(INTERNAL_NOTES_PATH)
products = load_json(PRODUCTS_PATH)

news, detected_encoding, detected_separator = read_news_csv(NEWS_PATH)


print(f"Кодировка CSV: {detected_encoding}")
print(f"Разделитель CSV: {repr(detected_separator)}")
print(f"Количество новостей: {len(news):,}")
print(f"Количество клиентов: {len(clients)}")
print(f"Количество внутренних заметок: {len(internal_notes)}")
print(f"Количество продуктов: {len(products)}")

display(news.head())

Кодировка CSV: utf-8-sig
Разделитель CSV: ','
Количество новостей: 1,880
Количество клиентов: 8
Количество внутренних заметок: 8
Количество продуктов: 10


,title,text,source
0,Начался пожар после взрыва на крупнейшем в мир...,Начался пожар после взрыва на крупнейшем в мир...,https://t.me/s/banksta
1,Отдых россиян в Азии снизил инфляцию в России.,Отдых россиян в Азии снизил инфляцию в России....,https://t.me/s/bankrollo
2,Правительство подготовило новую версию законоп...,Правительство подготовило новую версию законоп...,https://t.me/s/rbc_news
3,✨ Закрыт сбор на мобильные огневые группы!,✨ Закрыт сбор на мобильные огневые группы! ...,https://t.me/s/dva_majors
4,"↖️ Индия, один из ключевых покупателей российс...","↖️ Индия, один из ключевых покупателей российс...",https://t.me/s/kommersant


In [3]:
REQUIRED_COLUMNS = ["title", "text", "source"]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in news.columns
]

if missing_columns:
    raise ValueError(
        f"В news.csv отсутствуют колонки: {missing_columns}"
    )


# Оставляем только нужные колонки.
# Остальные колонки, если они есть, пока не удаляем,
# чтобы не потерять потенциально полезные данные.
news = news.copy()

for column in REQUIRED_COLUMNS:
    news[column] = news[column].fillna("").astype(str)


# Уникальный идентификатор строки
news.insert(
    0,
    "news_id",
    np.arange(1, len(news) + 1),
)


print("Пропуски после заполнения:")
display(news[REQUIRED_COLUMNS].isna().sum().to_frame("missing_count"))

print("Пустые значения:")
display(
    pd.DataFrame(
        {
            column: news[column].str.strip().eq("").sum()
            for column in REQUIRED_COLUMNS
        },
        index=["empty_count"],
    ).T
)

Пропуски после заполнения:


,missing_count
title,0
text,0
source,0


Пустые значения:


,empty_count
title,0
text,0
source,0


In [4]:
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
WHITESPACE_PATTERN = re.compile(r"\s+")


def clean_display_text(value: str) -> str:
    """
    Очищает текст от HTML, неразрывных пробелов
    и повторяющихся пробельных символов.
    """

    if value is None:
        return ""

    value = str(value)
    value = html.unescape(value)
    value = unicodedata.normalize("NFKC", value)

    value = value.replace("\xa0", " ")
    value = HTML_TAG_PATTERN.sub(" ", value)
    value = WHITESPACE_PATTERN.sub(" ", value)

    return value.strip()


def normalize_for_matching(value: str) -> str:
    """
    Нормализация для словарного поиска:
    - нижний регистр;
    - ё -> е;
    - удаление пунктуации;
    - схлопывание пробелов.
    """

    value = clean_display_text(value)
    value = value.lower().replace("ё", "е")

    # Оставляем русские и латинские буквы, цифры и пробелы
    value = re.sub(
        r"[^a-zа-я0-9]+",
        " ",
        value,
        flags=re.IGNORECASE,
    )

    value = WHITESPACE_PATTERN.sub(" ", value)

    return value.strip()


news["title_clean"] = news["title"].map(clean_display_text)
news["text_clean"] = news["text"].map(clean_display_text)
news["source_clean"] = news["source"].map(clean_display_text)

news["title_normalized"] = news["title_clean"].map(
    normalize_for_matching
)
news["text_normalized"] = news["text_clean"].map(
    normalize_for_matching
)


# Текст для будущей семантической модели.
# Заголовок повторяется дважды, чтобы иметь больший вес.
news["model_text"] = (
    news["title_clean"]
    + ". "
    + news["title_clean"]
    + ". "
    + news["text_clean"].str.slice(0, 3000)
)


display(
    news[
        [
            "news_id",
            "title_clean",
            "source_clean",
            "model_text",
        ]
    ].head()
)

,news_id,title_clean,source_clean,model_text
0,1,Начался пожар после взрыва на крупнейшем в мир...,https://t.me/s/banksta,Начался пожар после взрыва на крупнейшем в мир...
1,2,Отдых россиян в Азии снизил инфляцию в России.,https://t.me/s/bankrollo,Отдых россиян в Азии снизил инфляцию в России....
2,3,Правительство подготовило новую версию законоп...,https://t.me/s/rbc_news,Правительство подготовило новую версию законоп...
3,4,✨ Закрыт сбор на мобильные огневые группы!,https://t.me/s/dva_majors,✨ Закрыт сбор на мобильные огневые группы!. ✨ ...
4,5,"↖️ Индия, один из ключевых покупателей российс...",https://t.me/s/kommersant,"↖️ Индия, один из ключевых покупателей российс..."


In [5]:
COMPANY_ALIASES = {
    "РусГидро": [
        "РусГидро",
        "Рус Гидро",
        "ПАО РусГидро",
        "ПАО Рус Гидро",
        "Rushydro",
    ],

    "Аэрофлот": [
        "Аэрофлот",
        "Группа Аэрофлот",
        "ПАО Аэрофлот",
        "Aeroflot",
    ],

    "Норникель": [
        "Норникель",
        "Норильский никель",
        "ГМК Норильский никель",
        "ГМК Норникель",
        "Nornickel",
        "Norilsk Nickel",
    ],

    "РЖД": [
        "РЖД",
        "ОАО РЖД",
        "Российские железные дороги",
        "Russian Railways",
    ],

    "Магнит": [
        "Магнит",
        "ПАО Магнит",
        "Розничная сеть Магнит",
        "Magnit",
    ],

    "Лукойл": [
        "Лукойл",
        "ПАО Лукойл",
        "Lukoil",
    ],

    "МТС": [
        "МТС",
        "ПАО МТС",
        "Мобильные ТелеСистемы",
        "Мобильные телесистемы",
        "Mobile TeleSystems",
    ],

    "ФосАгро": [
        "ФосАгро",
        "Фос Агро",
        "ПАО ФосАгро",
        "PhosAgro",
        "Phos Agro",
    ],
}


# Проверяем, что для всех компаний из clients.json
# определены алиасы.
clients_without_aliases = set(clients) - set(COMPANY_ALIASES)

if clients_without_aliases:
    raise ValueError(
        "Для следующих клиентов не заданы алиасы: "
        f"{sorted(clients_without_aliases)}"
    )


print("Компании из clients.json:")
for company in clients:
    print(f"— {company}")

Компании из clients.json:
— РусГидро
— Аэрофлот
— Норникель
— РЖД
— Магнит
— Лукойл
— МТС
— ФосАгро


In [6]:
def alias_to_regex(alias: str) -> str:
    """
    Превращает алиас компании в часть регулярного выражения.
    Пробелы между словами допускаются в произвольном количестве.
    """

    normalized_alias = normalize_for_matching(alias)

    words = normalized_alias.split()

    escaped_words = [
        re.escape(word)
        for word in words
    ]

    return r"\s+".join(escaped_words)


def build_company_pattern(aliases: list[str]) -> re.Pattern:
    normalized_aliases = {
        normalize_for_matching(alias)
        for alias in aliases
    }

    normalized_aliases = sorted(
        normalized_aliases,
        key=len,
        reverse=True,
    )

    aliases_regex = [
        alias_to_regex(alias)
        for alias in normalized_aliases
    ]

    combined = "|".join(aliases_regex)

    # Вместо \b используем собственные границы,
    # поскольку они надёжнее работают со смешением
    # кириллицы, латиницы и цифр.
    pattern = (
        rf"(?<![a-zа-я0-9])"
        rf"(?:{combined})"
        rf"(?![a-zа-я0-9])"
    )

    return re.compile(
        pattern,
        flags=re.IGNORECASE,
    )


COMPANY_PATTERNS = {
    company: build_company_pattern(aliases)
    for company, aliases in COMPANY_ALIASES.items()
}

In [7]:
TITLE_WEIGHT = 5.0
LEAD_WEIGHT = 2.0
BODY_WEIGHT = 1.0

LEAD_LENGTH = 700


# У слова "магнит" есть обычное значение,
# поэтому для этой компании дополнительно проверяем
# наличие корпоративного контекста.
MAGNIT_CONTEXT_TERMS = {
    "компания",
    "ритейл",
    "ритейлер",
    "розничная",
    "розничный",
    "магазин",
    "торговая сеть",
    "выручка",
    "дивиденды",
    "акции",
    "поставщики",
    "покупатели",
    "сеть магазинов",
}


def count_matches(
    pattern: re.Pattern,
    text: str,
) -> int:
    if not text:
        return 0

    return len(pattern.findall(text))


def score_company_in_news(
    company: str,
    pattern: re.Pattern,
    title: str,
    body: str,
) -> dict:
    lead = body[:LEAD_LENGTH]
    body_tail = body[LEAD_LENGTH:]

    title_count = count_matches(pattern, title)
    lead_count = count_matches(pattern, lead)
    body_count = count_matches(pattern, body_tail)

    score = (
        TITLE_WEIGHT * title_count
        + LEAD_WEIGHT * lead_count
        + BODY_WEIGHT * body_count
    )

    # Защита от ложного определения компании "Магнит"
    # в научных или технических текстах.
    if company == "Магнит" and score > 0:
        context_text = f"{title} {lead}"

        has_company_context = any(
            term in context_text
            for term in MAGNIT_CONTEXT_TERMS
        )

        # Если корпоративного контекста нет,
        # значительно уменьшаем score.
        if not has_company_context:
            score *= 0.25

    return {
        "company": company,
        "score": float(score),
        "title_count": int(title_count),
        "lead_count": int(lead_count),
        "body_count": int(body_count),
    }


def get_company_confidence(
    best_score: float,
    second_score: float,
) -> str:
    """
    Простая интерпретируемая оценка уверенности.
    Позже её можно заменить обученной моделью.
    """

    score_gap = best_score - second_score

    if best_score >= 5 and score_gap >= 2:
        return "high"

    if best_score >= 2 and score_gap >= 1:
        return "medium"

    return "low"


def extract_company(
    title_normalized: str,
    text_normalized: str,
) -> dict:
    candidates = []

    for company, pattern in COMPANY_PATTERNS.items():
        company_result = score_company_in_news(
            company=company,
            pattern=pattern,
            title=title_normalized,
            body=text_normalized,
        )

        if company_result["score"] > 0:
            candidates.append(company_result)

    candidates.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    if not candidates:
        return {
            "company": None,
            "company_score": 0.0,
            "company_confidence": "not_found",
            "company_candidates": [],
            "company_evidence": {},
        }

    best_candidate = candidates[0]

    second_score = (
        candidates[1]["score"]
        if len(candidates) > 1
        else 0.0
    )

    confidence = get_company_confidence(
        best_score=best_candidate["score"],
        second_score=second_score,
    )

    company_candidates = [
        candidate["company"]
        for candidate in candidates
    ]

    company_evidence = {
        candidate["company"]: {
            "score": candidate["score"],
            "title_count": candidate["title_count"],
            "lead_count": candidate["lead_count"],
            "body_count": candidate["body_count"],
        }
        for candidate in candidates
    }

    return {
        "company": best_candidate["company"],
        "company_score": best_candidate["score"],
        "company_confidence": confidence,
        "company_candidates": company_candidates,
        "company_evidence": company_evidence,
    }

In [8]:
company_results = news.apply(
    lambda row: extract_company(
        title_normalized=row["title_normalized"],
        text_normalized=row["text_normalized"],
    ),
    axis=1,
)

company_results_df = pd.DataFrame(
    company_results.tolist(),
    index=news.index,
)

news = pd.concat(
    [
        news,
        company_results_df,
    ],
    axis=1,
)


display(
    news[
        [
            "news_id",
            "title_clean",
            "source_clean",
            "company",
            "company_score",
            "company_confidence",
            "company_candidates",
        ]
    ].head(20)
)

,news_id,title_clean,source_clean,company,company_score,company_confidence,company_candidates
0,1,Начался пожар после взрыва на крупнейшем в мир...,https://t.me/s/banksta,None,0.0,not_found,[]
1,2,Отдых россиян в Азии снизил инфляцию в России.,https://t.me/s/bankrollo,None,0.0,not_found,[]
2,3,Правительство подготовило новую версию законоп...,https://t.me/s/rbc_news,None,0.0,not_found,[]
3,4,✨ Закрыт сбор на мобильные огневые группы!,https://t.me/s/dva_majors,None,0.0,not_found,[]
4,5,"↖️ Индия, один из ключевых покупателей российс...",https://t.me/s/kommersant,None,0.0,not_found,[]
5,6,Wildberries & Russ отложила планы по развитию ...,https://t.me/s/kommersant,None,0.0,not_found,[]
6,7,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,https://t.me/s/kommersant,None,0.0,not_found,[]
7,8,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,https://t.me/s/markettwits,None,0.0,not_found,[]
8,9,😀 РынкиДеньгиВласть — это аналитический сервис...,https://t.me/s/AK47pfl,None,0.0,not_found,[]
9,10,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,https://t.me/s/markettwits,None,0.0,not_found,[]


In [9]:
COMPANY_DEFAULT_INDUSTRY = {
    "РусГидро": "энергетика",
    "Аэрофлот": "транспорт",
    "Норникель": "металлургия",
    "РЖД": "транспорт",
    "Магнит": "потребительский сектор",
    "Лукойл": "нефтегаз",
    "МТС": "телекоммуникации",
    "ФосАгро": "промышленность",
}


allowed_industries = {
    "экономика",
    "политика",
    "недвижимость",
    "транспорт",
    "промышленность",
    "потребительский сектор",
    "металлургия",
    "нефтегаз",
    "телекоммуникации",
    "энергетика",
    "строительство",
    "сельское хозяйство",
    "наука",
    "экология",
}


unknown_industries = (
    set(COMPANY_DEFAULT_INDUSTRY.values())
    - allowed_industries
)

if unknown_industries:
    raise ValueError(
        "В COMPANY_DEFAULT_INDUSTRY обнаружены "
        f"недопустимые отрасли: {unknown_industries}"
    )


news["company_industry_prior"] = news["company"].map(
    COMPANY_DEFAULT_INDUSTRY
)


display(
    news[
        [
            "news_id",
            "title_clean",
            "company",
            "company_industry_prior",
            "company_confidence",
        ]
    ].head(20)
)

,news_id,title_clean,company,company_industry_prior,company_confidence
0,1,Начался пожар после взрыва на крупнейшем в мир...,None,NaN,not_found
1,2,Отдых россиян в Азии снизил инфляцию в России.,None,NaN,not_found
2,3,Правительство подготовило новую версию законоп...,None,NaN,not_found
3,4,✨ Закрыт сбор на мобильные огневые группы!,None,NaN,not_found
4,5,"↖️ Индия, один из ключевых покупателей российс...",None,NaN,not_found
5,6,Wildberries & Russ отложила планы по развитию ...,None,NaN,not_found
6,7,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,None,NaN,not_found
7,8,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,None,NaN,not_found
8,9,😀 РынкиДеньгиВласть — это аналитический сервис...,None,NaN,not_found
9,10,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,None,NaN,not_found


In [10]:
company_statistics = (
    news["company"]
    .fillna("Компания не определена")
    .value_counts(dropna=False)
    .rename_axis("company")
    .reset_index(name="news_count")
)

company_statistics["share"] = (
    company_statistics["news_count"]
    / len(news)
)

display(company_statistics)


confidence_statistics = (
    news["company_confidence"]
    .value_counts(dropna=False)
    .rename_axis("confidence")
    .reset_index(name="news_count")
)

confidence_statistics["share"] = (
    confidence_statistics["news_count"]
    / len(news)
)

display(confidence_statistics)

,company,news_count,share
0,Компания не определена,1842,0.979787
1,Аэрофлот,17,0.009043
2,РЖД,6,0.003191
3,РусГидро,6,0.003191
4,Лукойл,4,0.002128
5,МТС,2,0.001064
6,Магнит,2,0.001064
7,Норникель,1,0.000532


,confidence,news_count,share
0,not_found,1842,0.979787
1,medium,18,0.009574
2,high,10,0.005319
3,low,10,0.005319


In [11]:
not_found = news.loc[
    news["company"].isna(),
    [
        "news_id",
        "title_clean",
        "source_clean",
        "text_clean",
    ],
].copy()

print(
    f"Компания не найдена для "
    f"{len(not_found)} из {len(news)} новостей "
    f"({len(not_found) / max(len(news), 1):.1%})."
)

display(not_found.head(30))

Компания не найдена для 1842 из 1880 новостей (98.0%).


,news_id,title_clean,source_clean,text_clean
0,1,Начался пожар после взрыва на крупнейшем в мир...,https://t.me/s/banksta,Начался пожар после взрыва на крупнейшем в мир...
1,2,Отдых россиян в Азии снизил инфляцию в России.,https://t.me/s/bankrollo,Отдых россиян в Азии снизил инфляцию в России....
2,3,Правительство подготовило новую версию законоп...,https://t.me/s/rbc_news,Правительство подготовило новую версию законоп...
3,4,✨ Закрыт сбор на мобильные огневые группы!,https://t.me/s/dva_majors,✨ Закрыт сбор на мобильные огневые группы! ✨ З...
4,5,"↖️ Индия, один из ключевых покупателей российс...",https://t.me/s/kommersant,"↖️ Индия, один из ключевых покупателей российс..."
5,6,Wildberries & Russ отложила планы по развитию ...,https://t.me/s/kommersant,Wildberries & Russ отложила планы по развитию ...
6,7,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,https://t.me/s/kommersant,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...
7,8,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,https://t.me/s/markettwits,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026. 📬 mt ...
8,9,😀 РынкиДеньгиВласть — это аналитический сервис...,https://t.me/s/AK47pfl,😀 РынкиДеньгиВласть — это аналитический сервис...
9,10,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,https://t.me/s/markettwits,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...


In [12]:
uncertain_news = news.loc[
    news["company_confidence"].isin(["low", "medium"]),
    [
        "news_id",
        "title_clean",
        "source_clean",
        "company",
        "company_score",
        "company_confidence",
        "company_candidates",
        "company_evidence",
    ],
].sort_values(
    by="company_score",
    ascending=True,
)

display(uncertain_news.head(50))

,news_id,title_clean,source_clean,company,company_score,company_confidence,company_candidates,company_evidence
328,329,🛍 Подборка карт с кэшбеком на маркетплейсах: п...,https://t.me/s/MarketOverview,Магнит,0.25,low,[Магнит],"{'Магнит': {'score': 0.25, 'title_count': 0, '..."
54,55,"💰 История о том, как Росимущество ЮГК продавал...",https://t.me/s/if_stocks,РЖД,1.00,low,[РЖД],"{'РЖД': {'score': 1.0, 'title_count': 0, 'lead..."
1534,1535,Нижний Новгород.,https://www.militarynews.ru,Лукойл,1.00,low,[Лукойл],"{'Лукойл': {'score': 1.0, 'title_count': 0, 'l..."
1350,1351,"Во вторник, 23 июня, ожидаются выплаты купонны...",https://rusbonds.ru/rss.asp,РЖД,1.00,low,"[РЖД, Магнит]","{'РЖД': {'score': 1.0, 'title_count': 0, 'lead..."
1082,1083,"""В связи с проведением в сентябре ремонтных ра...",https://rg.ru/tema/ekonomika/transport/rail,РЖД,1.00,low,[РЖД],"{'РЖД': {'score': 1.0, 'title_count': 0, 'lead..."
934,935,"Авиакомпанию ""Уральские авиалинии"" привлекли к...",http://www.m24.ru/rss.xml,Аэрофлот,1.00,low,[Аэрофлот],"{'Аэрофлот': {'score': 1.0, 'title_count': 0, ..."
810,811,Источник: «Почта России» Корпоративный мессенд...,https://retail.ru/rbc/pressreleases/,РЖД,1.00,low,[РЖД],"{'РЖД': {'score': 1.0, 'title_count': 0, 'lead..."
755,756,Почти 60 дронов сбито на подлете к Москве,https://news.mail.ru/rss/politics/90/,Аэрофлот,1.00,low,[Аэрофлот],"{'Аэрофлот': {'score': 1.0, 'title_count': 0, ..."
861,862,"Авиакомпания ""Россия"" переносит время вылета, ...",http://www.m24.ru/rss.xml,Аэрофлот,1.00,low,[Аэрофлот],"{'Аэрофлот': {'score': 1.0, 'title_count': 0, ..."
1092,1093,ДРСК подключила к электросетям около 300 новых...,http://energyland.info/news/rss,РусГидро,2.00,medium,[РусГидро],"{'РусГидро': {'score': 2.0, 'title_count': 0, ..."


In [13]:
multiple_companies = news.loc[
    news["company_candidates"].map(len) > 1,
    [
        "news_id",
        "title_clean",
        "source_clean",
        "company",
        "company_candidates",
        "company_evidence",
    ],
].copy()

print(
    f"Новостей с упоминанием нескольких клиентов: "
    f"{len(multiple_companies)}"
)

display(multiple_companies.head(30))

Новостей с упоминанием нескольких клиентов: 2


,news_id,title_clean,source_clean,company,company_candidates,company_evidence
1102,1103,Более 20 лет компания ARTW является одним из в...,https://companies.rbc.ru/news/,Лукойл,"[Лукойл, МТС]","{'Лукойл': {'score': 2.0, 'title_count': 0, 'l..."
1350,1351,"Во вторник, 23 июня, ожидаются выплаты купонны...",https://rusbonds.ru/rss.asp,РЖД,"[РЖД, Магнит]","{'РЖД': {'score': 1.0, 'title_count': 0, 'lead..."


In [14]:
stage1_output = news.copy()

stage1_output["company_candidates"] = (
    stage1_output["company_candidates"]
    .map(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
        )
    )
)

stage1_output["company_evidence"] = (
    stage1_output["company_evidence"]
    .map(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
        )
    )
)


STAGE1_OUTPUT_PATH = OUTPUT_DIR / "news_stage1_companies.csv"

stage1_output.to_csv(
    STAGE1_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Результат сохранён:\n{STAGE1_OUTPUT_PATH}")

Результат сохранён:
D:\sber\outputs\news_stage1_companies.csv


## Дальше определение отрасли

In [15]:
%pip install -q sentence-transformers scikit-learn scipy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import json
import re

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

from IPython.display import display

C:\Users\Devadevam\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
TITLE_COLUMN = (
    "title_clean"
    if "title_clean" in news.columns
    else "title"
)

TEXT_COLUMN = (
    "text_clean"
    if "text_clean" in news.columns
    else "text"
)

SOURCE_COLUMN = (
    "source_clean"
    if "source_clean" in news.columns
    else "source"
)

# После предыдущего этапа компания должна находиться
# в target_client. Если переименование не выполнялось,
# используем company.
if "target_client" in news.columns:
    COMPANY_COLUMN = "target_client"
elif "company" in news.columns:
    COMPANY_COLUMN = "company"
else:
    raise ValueError(
        "В news не найдена колонка target_client или company."
    )


for column in [
    TITLE_COLUMN,
    TEXT_COLUMN,
    SOURCE_COLUMN,
    COMPANY_COLUMN,
]:
    if column not in news.columns:
        raise ValueError(f"Не найдена колонка: {column}")


news[TITLE_COLUMN] = (
    news[TITLE_COLUMN]
    .fillna("")
    .astype(str)
)

news[TEXT_COLUMN] = (
    news[TEXT_COLUMN]
    .fillna("")
    .astype(str)
)

news[SOURCE_COLUMN] = (
    news[SOURCE_COLUMN]
    .fillna("")
    .astype(str)
)

print("Колонка заголовка:", TITLE_COLUMN)
print("Колонка текста:", TEXT_COLUMN)
print("Колонка источника:", SOURCE_COLUMN)
print("Колонка компании:", COMPANY_COLUMN)

Колонка заголовка: title_clean
Колонка текста: text_clean
Колонка источника: source_clean
Колонка компании: company


In [18]:
INDUSTRY_PROTOTYPES = {
    "экономика": [
        "Макроэкономические новости об инфляции, ВВП, бюджете, налогах и ключевой ставке.",
        "Изменение курса рубля, процентных ставок, государственных расходов и экономических показателей.",
        "Финансовый рынок, банки, инвестиции, денежно-кредитная политика и состояние экономики.",
        "Доходы населения, безработица, инфляция и экономический рост.",
    ],

    "политика": [
        "Решения органов государственной власти, правительства, парламента и президента.",
        "Выборы, политические партии, международные отношения, санкции и дипломатия.",
        "Принятие законов, государственное управление и политические заявления.",
        "Международная политика, переговоры государств и деятельность чиновников.",
    ],

    "недвижимость": [
        "Покупка, продажа и аренда квартир, домов и коммерческой недвижимости.",
        "Цены на жильё, ипотека, девелоперы и рынок недвижимости.",
        "Жилые комплексы, офисная недвижимость, склады и торговые площади.",
        "Спрос и предложение на первичном и вторичном рынке жилья.",
    ],

    "транспорт": [
        "Авиационные, железнодорожные, автомобильные и морские перевозки.",
        "Авиакомпании, аэропорты, железные дороги, автомобили и общественный транспорт.",
        "Логистика, грузоперевозки, пассажирские перевозки и транспортная инфраструктура.",
        "Работа поездов, самолётов, портов, дорог и транспортных компаний.",
    ],

    "промышленность": [
        "Промышленное производство, заводы, фабрики и выпуск оборудования.",
        "Машиностроение, химическая промышленность и производственные предприятия.",
        "Изменение объёмов промышленного производства и загрузки предприятий.",
        "Производство техники, оборудования, химической и другой промышленной продукции.",
    ],

    "потребительский сектор": [
        "Розничная торговля, магазины, рестораны и товары повседневного спроса.",
        "Продажи продуктов питания, одежды, бытовой техники и потребительских товаров.",
        "Ритейлеры, торговые сети, потребительский спрос и электронная коммерция.",
        "Покупательская активность, услуги населению и развитие розничных сетей.",
    ],

    "металлургия": [
        "Добыча и производство стали, алюминия, меди, никеля и других металлов.",
        "Металлургические предприятия, производство проката и обработка руды.",
        "Цены на металлы, горнодобывающие компании и металлургические комбинаты.",
        "Производство цветных и чёрных металлов.",
    ],

    "нефтегаз": [
        "Добыча, переработка и экспорт нефти и природного газа.",
        "Нефтяные и газовые месторождения, трубопроводы и нефтепродукты.",
        "Цены на нефть, газ, бензин, СПГ и деятельность нефтегазовых компаний.",
        "Нефтеперерабатывающие заводы, экспорт топлива и добыча углеводородов.",
    ],

    "телекоммуникации": [
        "Мобильная связь, интернет, телекоммуникационные компании и операторы связи.",
        "Сотовые сети, абоненты, тарифы, передача данных и телекоммуникационная инфраструктура.",
        "Развитие сетей связи, 5G, широкополосного интернета и цифровых сервисов.",
        "Операторы мобильной связи и интернет-провайдеры.",
    ],

    "энергетика": [
        "Производство и передача электрической и тепловой энергии.",
        "Электростанции, энергосети, генерация электроэнергии и энергокомпании.",
        "Атомная, гидроэнергетическая, тепловая и возобновляемая энергетика.",
        "Тарифы на электроэнергию, энергоблоки и энергетическая инфраструктура.",
    ],

    "строительство": [
        "Строительство зданий, дорог, мостов и инфраструктурных объектов.",
        "Строительные компании, подрядчики, строительные материалы и проекты.",
        "Возведение объектов, капитальный ремонт и развитие городской инфраструктуры.",
        "Строительство заводов, дорог, жилых и общественных объектов.",
    ],

    "сельское хозяйство": [
        "Производство зерна, овощей, мяса, молока и другой сельскохозяйственной продукции.",
        "Урожай, посевная, животноводство, растениеводство и фермерские хозяйства.",
        "Агропромышленный комплекс, удобрения, сельхозтехника и экспорт продовольствия.",
        "Состояние сельского хозяйства, зерновой рынок и производство продуктов питания.",
    ],

    "наука": [
        "Научные исследования, открытия, эксперименты и новые технологии.",
        "Работа учёных, университетов, лабораторий и исследовательских институтов.",
        "Физика, химия, биология, медицина, космос и научные разработки.",
        "Результаты научных исследований и публикации учёных.",
    ],

    "экология": [
        "Охрана природы, загрязнение воздуха и воды, климат и окружающая среда.",
        "Выбросы, отходы, экологический ущерб и природоохранные проекты.",
        "Изменение климата, лесные пожары, загрязнение и переработка отходов.",
        "Экологические требования, снижение выбросов и восстановление природы.",
    ],
}

INDUSTRIES = list(INDUSTRY_PROTOTYPES)

print("Количество отраслей:", len(INDUSTRIES))
print(INDUSTRIES)

Количество отраслей: 14
['экономика', 'политика', 'недвижимость', 'транспорт', 'промышленность', 'потребительский сектор', 'металлургия', 'нефтегаз', 'телекоммуникации', 'энергетика', 'строительство', 'сельское хозяйство', 'наука', 'экология']


In [19]:
INDUSTRY_KEYWORDS = {
    "экономика": [
        "инфляция",
        "ввп",
        "ключевая ставка",
        "центробанк",
        "банк россии",
        "бюджет",
        "налоги",
        "налог",
        "дефицит бюджета",
        "курс рубля",
        "денежно кредитная политика",
        "безработица",
        "экономический рост",
        "фондовый рынок",
        "облигации",
    ],

    "политика": [
        "президент",
        "правительство",
        "госдума",
        "совет федерации",
        "министр",
        "выборы",
        "депутат",
        "законопроект",
        "санкции",
        "переговоры",
        "международные отношения",
        "политическая партия",
        "дипломатия",
    ],

    "недвижимость": [
        "недвижимость",
        "квартира",
        "жилье",
        "ипотека",
        "аренда",
        "жилой комплекс",
        "новостройка",
        "девелопер",
        "квадратный метр",
        "коммерческая недвижимость",
        "рынок жилья",
    ],

    "транспорт": [
        "авиакомпания",
        "самолет",
        "аэропорт",
        "рейс",
        "железная дорога",
        "поезд",
        "ржд",
        "автомобиль",
        "автобус",
        "метро",
        "перевозка",
        "логистика",
        "грузоперевозка",
        "порт",
        "судно",
    ],

    "промышленность": [
        "промышленность",
        "завод",
        "фабрика",
        "производство",
        "машиностроение",
        "оборудование",
        "производственная линия",
        "промышленное предприятие",
        "выпуск продукции",
        "химическая промышленность",
    ],

    "потребительский сектор": [
        "ритейлер",
        "торговая сеть",
        "магазин",
        "розничная торговля",
        "потребительский спрос",
        "продажи",
        "ресторан",
        "супермаркет",
        "маркетплейс",
        "электронная коммерция",
        "товары повседневного спроса",
    ],

    "металлургия": [
        "металлургия",
        "сталь",
        "алюминий",
        "медь",
        "никель",
        "палладий",
        "платина",
        "металл",
        "руда",
        "металлургический комбинат",
        "горнодобывающая компания",
        "прокат",
    ],

    "нефтегаз": [
        "нефть",
        "газ",
        "нефтегаз",
        "баррель",
        "нефтепровод",
        "газопровод",
        "месторождение",
        "нефтепродукт",
        "бензин",
        "дизель",
        "спг",
        "нефтепереработка",
        "добыча углеводородов",
    ],

    "телекоммуникации": [
        "телекоммуникации",
        "оператор связи",
        "мобильная связь",
        "сотовая связь",
        "абонент",
        "интернет провайдер",
        "телеком",
        "тариф",
        "5g",
        "сеть связи",
        "передача данных",
    ],

    "энергетика": [
        "энергетика",
        "электроэнергия",
        "электростанция",
        "энергоблок",
        "генерация",
        "энергосеть",
        "гидроэлектростанция",
        "аэс",
        "тэс",
        "гэc",
        "возобновляемая энергия",
        "мощность электростанции",
    ],

    "строительство": [
        "строительство",
        "строительный объект",
        "подрядчик",
        "стройка",
        "мост",
        "дорога",
        "инфраструктурный объект",
        "капитальный ремонт",
        "строительные материалы",
        "возведение",
    ],

    "сельское хозяйство": [
        "сельское хозяйство",
        "урожай",
        "зерно",
        "пшеница",
        "посевная",
        "фермер",
        "животноводство",
        "растениеводство",
        "удобрения",
        "агропромышленный комплекс",
        "сельхозпродукция",
        "экспорт зерна",
    ],

    "наука": [
        "ученые",
        "исследование",
        "научное открытие",
        "эксперимент",
        "лаборатория",
        "университет",
        "научный институт",
        "исследователи",
        "технология",
        "космос",
        "физика",
        "биология",
    ],

    "экология": [
        "экология",
        "экологический",
        "окружающая среда",
        "выбросы",
        "загрязнение",
        "отходы",
        "климат",
        "парниковые газы",
        "природоохранный",
        "утилизация",
        "лесной пожар",
        "разлив нефти",
    ],
}

In [20]:
INDUSTRY_KEYWORDS["энергетика"] = [
    word.replace("гэc", "гэс")
    for word in INDUSTRY_KEYWORDS["энергетика"]
]

In [21]:
COMPANY_INDUSTRY_PRIOR = {
    "РусГидро": "энергетика",
    "Аэрофлот": "транспорт",
    "Норникель": "металлургия",
    "РЖД": "транспорт",
    "Магнит": "потребительский сектор",
    "Лукойл": "нефтегаз",
    "МТС": "телекоммуникации",
    "ФосАгро": "промышленность",
}

In [22]:
def normalize_industry_text(value: str) -> str:
    value = str(value or "")
    value = value.lower().replace("ё", "е")
    value = re.sub(r"[^a-zа-я0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


news["industry_text"] = (
    news[TITLE_COLUMN]
    + ". "
    + news[TITLE_COLUMN]
    + ". "
    + news[TEXT_COLUMN].str.slice(0, 1800)
)

news["industry_text_normalized"] = (
    news["industry_text"]
    .map(normalize_industry_text)
)

In [23]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Модель загружена:", EMBEDDING_MODEL_NAME)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5900.08it/s]


Модель загружена: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [24]:
prototype_texts = []
prototype_industries = []

for industry, descriptions in INDUSTRY_PROTOTYPES.items():
    for description in descriptions:
        prototype_texts.append(description)
        prototype_industries.append(industry)


prototype_embeddings = embedding_model.encode(
    prototype_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)


industry_centroids = []

for industry in INDUSTRIES:
    indices = [
        index
        for index, value in enumerate(prototype_industries)
        if value == industry
    ]

    centroid = prototype_embeddings[indices].mean(axis=0)

    centroid = centroid / (
        np.linalg.norm(centroid) + 1e-12
    )

    industry_centroids.append(centroid)


industry_centroids = np.asarray(
    industry_centroids,
    dtype=np.float32,
)

print(industry_centroids.shape)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  6.43it/s]

(14, 384)


In [25]:
industry_news_embeddings = embedding_model.encode(
    news["industry_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

industry_news_embeddings = np.asarray(
    industry_news_embeddings,
    dtype=np.float32,
)

semantic_industry_scores = (
    industry_news_embeddings
    @ industry_centroids.T
)

print(semantic_industry_scores.shape)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 59/59 [00:47<00:00,  1.24it/s]

(1880, 14)


In [26]:
def phrase_count(
    normalized_text: str,
    normalized_phrase: str,
) -> int:
    if not normalized_phrase:
        return 0

    pattern = (
        r"(?<![a-zа-я0-9])"
        + re.escape(normalized_phrase).replace(r"\ ", r"\s+")
        + r"(?![a-zа-я0-9])"
    )

    return len(
        re.findall(
            pattern,
            normalized_text,
            flags=re.IGNORECASE,
        )
    )


def get_keyword_scores(
    title: str,
    body: str,
) -> np.ndarray:
    title = normalize_industry_text(title)
    body = normalize_industry_text(body)

    scores = np.zeros(
        len(INDUSTRIES),
        dtype=np.float32,
    )

    for industry_index, industry in enumerate(INDUSTRIES):
        score = 0.0

        for keyword in INDUSTRY_KEYWORDS[industry]:
            keyword = normalize_industry_text(keyword)

            title_count = phrase_count(
                title,
                keyword,
            )

            body_count = phrase_count(
                body,
                keyword,
            )

            score += 2.5 * title_count
            score += 1.0 * body_count

        scores[industry_index] = score

    # Приводим к диапазону 0–1.
    scores = 1.0 - np.exp(-scores / 2.0)

    return scores

In [27]:
keyword_industry_scores = np.vstack(
    [
        get_keyword_scores(
            title=row[TITLE_COLUMN],
            body=row[TEXT_COLUMN][:1800],
        )
        for _, row in tqdm(
            news.iterrows(),
            total=len(news),
        )
    ]
)

print(keyword_industry_scores.shape)

100%|█████████████████████████████████████████████████████████████████████████████| 1880/1880 [00:12<00:00, 155.97it/s]

(1880, 14)


In [28]:
company_prior_scores = np.zeros(
    (
        len(news),
        len(INDUSTRIES),
    ),
    dtype=np.float32,
)


for row_position, company in enumerate(
    news[COMPANY_COLUMN]
):
    industry = COMPANY_INDUSTRY_PRIOR.get(company)

    if industry is None:
        continue

    industry_index = INDUSTRIES.index(industry)

    company_prior_scores[
        row_position,
        industry_index,
    ] = 1.0

In [29]:
SEMANTIC_WEIGHT = 1.00
KEYWORD_BONUS = 0.08
COMPANY_PRIOR_BONUS = 0.04


final_industry_scores = (
    SEMANTIC_WEIGHT * semantic_industry_scores
    + KEYWORD_BONUS * keyword_industry_scores
    + COMPANY_PRIOR_BONUS * company_prior_scores
)

In [30]:
def interpret_industry_confidence(
    best_score: float,
    second_score: float,
) -> str:
    margin = best_score - second_score

    if margin >= 0.08:
        return "high"

    if margin >= 0.03:
        return "medium"

    return "low"

In [31]:
industry_results = []

for row_index in range(len(news)):
    row_scores = final_industry_scores[row_index]

    sorted_indices = np.argsort(
        row_scores
    )[::-1]

    best_index = sorted_indices[0]
    second_index = sorted_indices[1]

    best_industry = INDUSTRIES[best_index]
    best_score = float(row_scores[best_index])
    second_score = float(row_scores[second_index])

    top_candidates = [
        {
            "industry": INDUSTRIES[index],
            "score": round(
                float(row_scores[index]),
                4,
            ),
        }
        for index in sorted_indices[:3]
    ]

    industry_results.append(
        {
            "industry": best_industry,
            "industry_score": best_score,
            "industry_margin": (
                best_score - second_score
            ),
            "industry_confidence": (
                interpret_industry_confidence(
                    best_score,
                    second_score,
                )
            ),
            "industry_candidates": top_candidates,
        }
    )


industry_results_df = pd.DataFrame(
    industry_results,
    index=news.index,
)

In [32]:
industry_output_columns = [
    "industry",
    "industry_score",
    "industry_margin",
    "industry_confidence",
    "industry_candidates",
]

news = news.drop(
    columns=[
        column
        for column in industry_output_columns
        if column in news.columns
    ],
    errors="ignore",
)

news = pd.concat(
    [
        news,
        industry_results_df,
    ],
    axis=1,
)

In [33]:
display(
    news[
        [
            "news_id",
            TITLE_COLUMN,
            COMPANY_COLUMN,
            "industry",
            "industry_score",
            "industry_margin",
            "industry_confidence",
            "industry_candidates",
        ]
    ].head(30)
)

,news_id,title_clean,company,industry,industry_score,industry_margin,industry_confidence,industry_candidates
0,1,Начался пожар после взрыва на крупнейшем в мир...,None,нефтегаз,0.522087,0.196130,high,"[{'industry': 'нефтегаз', 'score': 0.5221}, {'..."
1,2,Отдых россиян в Азии снизил инфляцию в России.,None,экономика,0.527688,0.275853,high,"[{'industry': 'экономика', 'score': 0.5277}, {..."
2,3,Правительство подготовило новую версию законоп...,None,политика,0.338848,0.117787,high,"[{'industry': 'политика', 'score': 0.3388}, {'..."
3,4,✨ Закрыт сбор на мобильные огневые группы!,None,телекоммуникации,0.287161,0.024337,low,"[{'industry': 'телекоммуникации', 'score': 0.2..."
4,5,"↖️ Индия, один из ключевых покупателей российс...",None,нефтегаз,0.540161,0.234390,high,"[{'industry': 'нефтегаз', 'score': 0.5402}, {'..."
5,6,Wildberries & Russ отложила планы по развитию ...,None,потребительский сектор,0.097967,0.008933,low,"[{'industry': 'потребительский сектор', 'score..."
6,7,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,None,экономика,0.323296,0.136059,high,"[{'industry': 'экономика', 'score': 0.3233}, {..."
7,8,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,None,экономика,0.367123,0.140842,high,"[{'industry': 'экономика', 'score': 0.3671}, {..."
8,9,😀 РынкиДеньгиВласть — это аналитический сервис...,None,потребительский сектор,0.457881,0.000783,low,"[{'industry': 'потребительский сектор', 'score..."
9,10,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,None,экономика,0.388662,0.240531,high,"[{'industry': 'экономика', 'score': 0.3887}, {..."


In [34]:
industry_statistics = (
    news["industry"]
    .value_counts()
    .rename_axis("industry")
    .reset_index(name="news_count")
)

industry_statistics["share"] = (
    industry_statistics["news_count"]
    / len(news)
)

display(industry_statistics)

,industry,news_count,share
0,экономика,339,0.180319
1,наука,259,0.137766
2,политика,207,0.110106
3,транспорт,188,0.100000
4,экология,159,0.084574
5,строительство,151,0.080319
6,нефтегаз,150,0.079787
7,потребительский сектор,95,0.050532
8,промышленность,87,0.046277
9,недвижимость,77,0.040957


In [35]:
low_confidence_industries = news.loc[
    news["industry_confidence"] == "low",
    [
        "news_id",
        TITLE_COLUMN,
        COMPANY_COLUMN,
        "industry",
        "industry_margin",
        "industry_candidates",
    ],
].sort_values(
    "industry_margin",
    ascending=True,
)

display(low_confidence_industries.head(50))

,news_id,title_clean,company,industry,industry_margin,industry_candidates
657,658,В Тульской области прогремели взрывы на фоне а...,None,транспорт,0.000022,"[{'industry': 'транспорт', 'score': 0.245}, {'..."
1849,1850,Двое немцев попытались въехать в РФ с поддельн...,None,транспорт,0.000032,"[{'industry': 'транспорт', 'score': 0.1087}, {..."
866,867,Сергей Кельчевский из Екатеринбурга выиграл 89...,None,потребительский сектор,0.000154,"[{'industry': 'потребительский сектор', 'score..."
1161,1162,Машины не ехали: Казахстан пережил конвективны...,None,экология,0.000247,"[{'industry': 'экология', 'score': 0.2012}, {'..."
1065,1066,Победитель телевизионного проекта «Песни» Олег...,None,наука,0.000248,"[{'industry': 'наука', 'score': 0.1948}, {'ind..."
1838,1839,"По его словам, эффект охлаждения будет временным.",None,экология,0.000395,"[{'industry': 'экология', 'score': 0.235}, {'i..."
439,440,Над Москвой сбит беспилотник,None,транспорт,0.000423,"[{'industry': 'транспорт', 'score': 0.2543}, {..."
1191,1192,Жители Индии сняли на видео гигантский торнадо,None,строительство,0.000620,"[{'industry': 'строительство', 'score': 0.1077..."
823,824,В администрации Барнаула на совещании с перево...,None,строительство,0.000627,"[{'industry': 'строительство', 'score': 0.3637..."
531,532,"Суд в Подмосковье взыскал с ""турагента"" Хатуны...",None,телекоммуникации,0.000688,"[{'industry': 'телекоммуникации', 'score': 0.1..."


## Группировка новостей

In [36]:
news["event_text"] = (
    news[TITLE_COLUMN]
    + ". "
    + news[TITLE_COLUMN]
    + ". "
    + news[TITLE_COLUMN]
    + ". "
    + news[TEXT_COLUMN].str.slice(0, 1000)
)

In [37]:
event_embeddings = embedding_model.encode(
    news["event_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

event_embeddings = np.asarray(
    event_embeddings,
    dtype=np.float32,
)

semantic_event_similarity = (
    event_embeddings
    @ event_embeddings.T
)

print(semantic_event_similarity.shape)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 59/59 [00:45<00:00,  1.29it/s]

(1880, 1880)


In [38]:
title_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    max_features=50_000,
)

title_tfidf = title_vectorizer.fit_transform(
    news[TITLE_COLUMN]
    .fillna("")
    .astype(str)
)

lexical_title_similarity = (
    title_tfidf
    @ title_tfidf.T
).toarray().astype(np.float32)

print(lexical_title_similarity.shape)

(1880, 1880)


In [39]:
event_similarity = (
    0.80 * semantic_event_similarity
    + 0.20 * lexical_title_similarity
)

In [40]:
industry_values = (
    news["industry"]
    .fillna("")
    .astype(str)
    .to_numpy()
)

different_industry = (
    industry_values[:, None]
    != industry_values[None, :]
)

event_similarity[
    different_industry
] -= 0.07

In [41]:
company_values = (
    news[COMPANY_COLUMN]
    .fillna("")
    .astype(str)
    .to_numpy()
)

company_present = company_values != ""

both_have_company = (
    company_present[:, None]
    & company_present[None, :]
)

same_company = (
    company_values[:, None]
    == company_values[None, :]
)

different_company = (
    both_have_company
    & ~same_company
)

same_nonempty_company = (
    both_have_company
    & same_company
)

# Разные целевые компании — небольшой штраф.
event_similarity[
    different_company
] -= 0.04

# Одинаковая целевая компания — небольшой бонус.
event_similarity[
    same_nonempty_company
] += 0.03

In [42]:
normalized_titles = (
    news[TITLE_COLUMN]
    .map(normalize_industry_text)
    .to_numpy()
)

same_normalized_title = (
    normalized_titles[:, None]
    == normalized_titles[None, :]
)

nonempty_titles = normalized_titles != ""

same_nonempty_title = (
    same_normalized_title
    & nonempty_titles[:, None]
    & nonempty_titles[None, :]
)

event_similarity[
    same_nonempty_title
] = np.maximum(
    event_similarity[same_nonempty_title],
    0.99,
)

In [43]:
event_similarity = np.clip(
    event_similarity,
    0.0,
    1.0,
)

event_distance = (
    1.0 - event_similarity
)

np.fill_diagonal(
    event_distance,
    0.0,
)

In [44]:
EVENT_DISTANCE_THRESHOLD = 0.18

In [45]:
try:
    event_clusterer = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=EVENT_DISTANCE_THRESHOLD,
        compute_full_tree=True,
    )

except TypeError:
    # Совместимость со старыми версиями sklearn.
    event_clusterer = AgglomerativeClustering(
        n_clusters=None,
        affinity="precomputed",
        linkage="average",
        distance_threshold=EVENT_DISTANCE_THRESHOLD,
        compute_full_tree=True,
    )


raw_event_labels = event_clusterer.fit_predict(
    event_distance
)

print(
    "Количество событий:",
    len(np.unique(raw_event_labels)),
)

Количество событий: 1712


In [46]:
event_first_positions = {}

for position, label in enumerate(raw_event_labels):
    if label not in event_first_positions:
        event_first_positions[label] = position


ordered_labels = sorted(
    event_first_positions,
    key=event_first_positions.get,
)

event_label_mapping = {
    label: f"E{index:04d}"
    for index, label in enumerate(
        ordered_labels,
        start=1,
    )
}

news["event_id"] = [
    event_label_mapping[label]
    for label in raw_event_labels
]

In [47]:
SOURCE_ALIASES = {
    # При необходимости сюда можно добавить:
    # "риа новости": "РИА Новости",
    # "ria ru": "РИА Новости",
    # "рбк": "РБК",
    # "rbc ru": "РБК",
}


def normalize_source_name(source: str) -> str:
    source = str(source or "").strip()
    source = source.lower().replace("ё", "е")

    source = re.sub(
        r"https?://",
        "",
        source,
    )

    source = re.sub(
        r"^www\.",
        "",
        source,
    )

    source = source.split("/")[0]

    normalized_key = re.sub(
        r"[^a-zа-я0-9]+",
        " ",
        source,
    )

    normalized_key = re.sub(
        r"\s+",
        " ",
        normalized_key,
    ).strip()

    return SOURCE_ALIASES.get(
        normalized_key,
        normalized_key,
    )


news["source_normalized"] = (
    news[SOURCE_COLUMN]
    .map(normalize_source_name)
)

In [48]:
event_representative_titles = {}

for event_id, group in news.groupby("event_id"):
    positions = group.index.to_numpy()

    cluster_embeddings = event_embeddings[
        positions
    ]

    centroid = cluster_embeddings.mean(axis=0)

    centroid = centroid / (
        np.linalg.norm(centroid) + 1e-12
    )

    similarities = (
        cluster_embeddings
        @ centroid
    )

    best_local_position = int(
        np.argmax(similarities)
    )

    best_dataframe_index = positions[
        best_local_position
    ]

    representative_title = news.loc[
        best_dataframe_index,
        TITLE_COLUMN,
    ]

    event_representative_titles[
        event_id
    ] = representative_title

In [49]:
news = news.reset_index(drop=True)

In [50]:
event_statistics = (
    news.groupby("event_id")
    .agg(
        event_articles_count=(
            "news_id",
            "size",
        ),
        event_sources_count=(
            "source_normalized",
            "nunique",
        ),
    )
    .reset_index()
)

In [51]:
event_statistics["citation_count"] = (
    event_statistics["event_sources_count"]
    - 1
).clip(lower=0)

In [52]:
maximum_citation_count = (
    event_statistics["citation_count"].max()
)

if maximum_citation_count > 0:
    event_statistics["citation_index"] = (
        100.0
        * np.log1p(
            event_statistics["citation_count"]
        )
        / np.log1p(maximum_citation_count)
    )
else:
    event_statistics["citation_index"] = 0.0


event_statistics["citation_index"] = (
    event_statistics["citation_index"]
    .round(2)
)

In [53]:
event_statistics[
    "event_representative_title"
] = event_statistics["event_id"].map(
    event_representative_titles
)

In [54]:
event_sources = (
    news.groupby("event_id")[
        "source_normalized"
    ]
    .apply(
        lambda values: sorted(
            {
                value
                for value in values
                if value
            }
        )
    )
    .rename("event_sources")
    .reset_index()
)

event_statistics = event_statistics.merge(
    event_sources,
    on="event_id",
    how="left",
)

In [55]:
columns_to_remove = [
    "event_articles_count",
    "event_sources_count",
    "citation_count",
    "citation_index",
    "event_representative_title",
    "event_sources",
]

news = news.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in news.columns
    ],
    errors="ignore",
)

news = news.merge(
    event_statistics,
    on="event_id",
    how="left",
)

In [56]:
most_cited_events = (
    event_statistics
    .sort_values(
        [
            "citation_index",
            "event_sources_count",
            "event_articles_count",
        ],
        ascending=False,
    )
)

display(
    most_cited_events.head(30)
)

,event_id,event_articles_count,event_sources_count,citation_count,citation_index,event_representative_title,event_sources
391,E0392,7,7,6,100.00,Средний размер пенсионного обеспечения более 3...,"[kamchatka aif ru, m24 ru, news mail ru, osnme..."
490,E0491,6,5,4,82.71,В базовую программу ОМС включили 14 новых мето...,"[kolomna spravka ru, m24 ru, news mail ru, tvz..."
142,E0143,5,5,4,82.71,Федеральная антимонопольная служба (ФАС) сообщ...,"[akm ru, energyland info, pnp ru, t me, vm ru]"
1087,E1088,4,4,3,71.24,Ошибочное срабатывание датчика разгерметизации...,"[aviaport ru, krym aif ru, nsn fm, russian rt ..."
1354,E1355,4,4,3,71.24,По меньшей мере 13 человек погибли и 66 постра...,"[bel aif ru, life ru, m24 ru, sb by]"
24,E0025,7,3,2,56.46,В результате взрыва на заводе в Катаре пострад...,"[kolomna spravka ru, news mail ru, t me]"
129,E0130,4,3,2,56.46,ГОСА Мосбиржи не состоялось из-за отсутствия к...,"[m interfax ru, t me, vedomosti ru]"
180,E0181,3,3,2,56.46,Вице-премьер РФ Александр Новак поручил ведомс...,"[akm ru, ria ru, t me]"
372,E0373,3,3,2,56.46,Ненецкий (НАО) и Ямало-Ненецкий (ЯНАО) автоном...,"[life ru, ria ru, vm ru]"
429,E0430,3,3,2,56.46,Четырнадцать новых методов лечения вошли в баз...,"[osnmedia ru, ria ru, vm ru]"


In [57]:
if len(most_cited_events) > 0:
    example_event_id = (
        most_cited_events.iloc[0]["event_id"]
    )

    display(
        news.loc[
            news["event_id"] == example_event_id,
            [
                "news_id",
                TITLE_COLUMN,
                SOURCE_COLUMN,
                COMPANY_COLUMN,
                "industry",
                "event_id",
                "citation_index",
            ],
        ]
    )

,news_id,title_clean,source_clean,company,industry,event_id,citation_index
412,413,Средний размер пенсионного обеспечения более 3...,https://ria.ru/society/,None,экономика,E0392,100.0
461,462,Средняя пенсия неработающих граждан в мае этог...,http://www.m24.ru/rss.xml,None,экономика,E0392,100.0
463,464,Средний размер пенсий неработающих россиян в м...,https://www.osnmedia.ru/,None,экономика,E0392,100.0
486,487,В мае 2026 года средний размер пенсии среди не...,http://kamchatka.aif.ru/rss/all.php,None,экономика,E0392,100.0
538,539,В мае 2026 года средний размер пенсионных выпл...,http://yamal.aif.ru/rss/all.php,None,экономика,E0392,100.0
571,572,Средняя пенсия превысила 30 тысяч рублей в 13 ...,https://news.mail.ru/rss/,None,экономика,E0392,100.0
802,803,Средний размер пенсии неработающих граждан пре...,http://vm.ru/rss,None,экономика,E0392,100.0


In [58]:
cluster_preview = (
    news.groupby("event_id")
    .agg(
        article_count=("news_id", "size"),
        source_count=(
            "source_normalized",
            "nunique",
        ),
        industry=("industry", "first"),
        representative_title=(
            "event_representative_title",
            "first",
        ),
        titles=(
            TITLE_COLUMN,
            lambda values: list(values)[:10],
        ),
    )
    .sort_values(
        "article_count",
        ascending=False,
    )
)

display(cluster_preview.head(30))

,article_count,source_count,industry,representative_title,titles
event_id,,,,,
E0739,24,1,экология,"22 июня, Минск /Корр.","[22 июня, Минск /Корр., 22 июня, Минск /Корр.,..."
E0655,10,2,наука,Москва.,"[Москва., Москва., Москва., Москва., Москва., ..."
E0392,7,7,экономика,Средний размер пенсионного обеспечения более 3...,[Средний размер пенсионного обеспечения более ...
E0025,7,3,промышленность,В результате взрыва на заводе в Катаре пострад...,[При взрыве на заводе в Катаре пострадали мини...
E0491,6,5,наука,В базовую программу ОМС включили 14 новых мето...,[Четырнадцать новых методов лечения вошли в ба...
E0143,5,5,нефтегаз,Федеральная антимонопольная служба (ФАС) сообщ...,[ФАС взаимодействует с цифровыми платформами д...
E1088,4,4,транспорт,Ошибочное срабатывание датчика разгерметизации...,"[Самолёт авиакомпании «Победа», следовавший из..."
E1355,4,4,нефтегаз,По меньшей мере 13 человек погибли и 66 постра...,[Взрыв на газоперерабатывающем заводе в катарс...
E0130,4,3,политика,ГОСА Мосбиржи не состоялось из-за отсутствия к...,[ГОСА Мосбиржи не состоялось из-за отсутствия ...


In [59]:
news["_final_company"] = news[
    COMPANY_COLUMN
]

In [60]:
final_news = news[
    [
        "news_id",
        TITLE_COLUMN,
        TEXT_COLUMN,
        SOURCE_COLUMN,
        "_final_company",
        "industry",
        "industry_score",
        "industry_confidence",
        "event_id",
        "event_representative_title",
        "event_articles_count",
        "event_sources_count",
        "citation_count",
        "citation_index",
    ]
].copy()

In [61]:
final_news = final_news.rename(
    columns={
        TITLE_COLUMN: "title",
        TEXT_COLUMN: "text",
        SOURCE_COLUMN: "source",
        "_final_company": "company",
    }
)

In [62]:
display(final_news.head(30))

,news_id,title,text,source,company,industry,industry_score,industry_confidence,event_id,event_representative_title,event_articles_count,event_sources_count,citation_count,citation_index
0,1,Начался пожар после взрыва на крупнейшем в мир...,Начался пожар после взрыва на крупнейшем в мир...,https://t.me/s/banksta,None,нефтегаз,0.522087,high,E0001,Начался пожар после взрыва на крупнейшем в мир...,1,1,0,0.00
1,2,Отдых россиян в Азии снизил инфляцию в России.,Отдых россиян в Азии снизил инфляцию в России....,https://t.me/s/bankrollo,None,экономика,0.527688,high,E0002,Отдых россиян в Азии снизил инфляцию в России.,1,1,0,0.00
2,3,Правительство подготовило новую версию законоп...,Правительство подготовило новую версию законоп...,https://t.me/s/rbc_news,None,политика,0.338848,high,E0003,Правительство подготовило новую версию законоп...,1,1,0,0.00
3,4,✨ Закрыт сбор на мобильные огневые группы!,✨ Закрыт сбор на мобильные огневые группы! ✨ З...,https://t.me/s/dva_majors,None,телекоммуникации,0.287161,low,E0004,✨ Закрыт сбор на мобильные огневые группы!,1,1,0,0.00
4,5,"↖️ Индия, один из ключевых покупателей российс...","↖️ Индия, один из ключевых покупателей российс...",https://t.me/s/kommersant,None,нефтегаз,0.540161,high,E0005,"↖️ Индия, один из ключевых покупателей российс...",1,1,0,0.00
5,6,Wildberries & Russ отложила планы по развитию ...,Wildberries & Russ отложила планы по развитию ...,https://t.me/s/kommersant,None,потребительский сектор,0.097967,low,E0006,🤪 Wildberries заморозила развитие игрового нап...,2,1,0,0.00
6,7,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,https://t.me/s/kommersant,None,экономика,0.323296,high,E0007,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,1,1,0,0.00
7,8,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026. 📬 mt ...,https://t.me/s/markettwits,None,экономика,0.367123,high,E0008,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,1,1,0,0.00
8,9,😀 РынкиДеньгиВласть — это аналитический сервис...,😀 РынкиДеньгиВласть — это аналитический сервис...,https://t.me/s/AK47pfl,None,потребительский сектор,0.457881,low,E0009,😀 РынкиДеньгиВласть — это аналитический сервис...,1,1,0,0.00
9,10,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,https://t.me/s/markettwits,None,экономика,0.388662,high,E0010,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,1,1,0,0.00


In [63]:
FINAL_NEWS_PATH = (
    OUTPUT_DIR / "news_final.csv"
)

final_news.to_csv(
    FINAL_NEWS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Итоговый файл сохранён:\n{FINAL_NEWS_PATH}"
)

Итоговый файл сохранён:
D:\sber\outputs\news_final.csv


In [64]:
EVENTS_OUTPUT_PATH = (
    OUTPUT_DIR / "events.csv"
)

events_output = event_statistics.copy()

events_output["event_sources"] = (
    events_output["event_sources"]
    .map(
        lambda values: json.dumps(
            values,
            ensure_ascii=False,
        )
    )
)

events_output.to_csv(
    EVENTS_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Таблица событий сохранена:\n"
    f"{EVENTS_OUTPUT_PATH}"
)

Таблица событий сохранена:
D:\sber\outputs\events.csv


### тут у нас получилось, что малоинформативные заголовки типа "Москва" объединяются в один кластер, даже когда новость о разных событиях. Надо от такого избавиться

In [65]:
news = news.reset_index(drop=True)

In [66]:
GENERIC_TITLES = {
    "москва",
    "уфа",
    "россия",
    "мир",
    "общество",
    "экономика",
    "политика",
    "бизнес",
    "спорт",
    "наука",
    "новости",
    "главное",
    "события",
    "регионы",
    "страна",
    "республика",
    "санкт петербург",
    "нижний новгород",
}


def normalize_event_title(title: str) -> str:
    title = str(title or "")
    title = title.lower().replace("ё", "е")

    title = re.sub(
        r"[^a-zа-я0-9]+",
        " ",
        title,
    )

    title = re.sub(
        r"\s+",
        " ",
        title,
    )

    return title.strip()


def is_weak_event_title(title: str) -> bool:
    """
    Возвращает True, если заголовок практически
    ничего не сообщает о событии.
    """

    normalized = normalize_event_title(title)

    if not normalized:
        return True

    if normalized in GENERIC_TITLES:
        return True

    words = normalized.split()

    # Однословные заголовки вроде "Москва".
    if len(words) == 1:
        return True

    # Двухсловные заголовки вроде "Нижний Новгород".
    # Заголовок с числом может быть информативным:
    # "Инфляция 8,2%".
    if len(words) == 2 and not any(
        character.isdigit()
        for character in normalized
    ):
        return True

    return False


news["event_title_normalized"] = (
    news[TITLE_COLUMN]
    .map(normalize_event_title)
)

news["weak_event_title"] = (
    news[TITLE_COLUMN]
    .map(is_weak_event_title)
)


print(
    "Количество слабых заголовков:",
    news["weak_event_title"].sum(),
)

display(
    news.loc[
        news["weak_event_title"],
        [
            "news_id",
            TITLE_COLUMN,
            SOURCE_COLUMN,
        ],
    ].head(30)
)

Количество слабых заголовков: 59


,news_id,title_clean,source_clean
27,28,💱 Какой курс?,https://t.me/s/selfinvestor
155,156,"Инвесторы, запоминаем",https://t.me/s/cbrstocks
169,170,"#MVIDАкционеры ""М.",https://t.me/s/cbrstocks
189,190,15.,https://t.me/s/c0ldness
195,196,#MVIDАкционеры М.,https://t.me/s/cbrstocks
267,268,В ст.,https://t.me/s/dealsma
310,311,💥 После ставки.,https://t.me/s/gpb_investments
325,326,🎼 Инвесткомитет.,https://t.me/s/gpb_investments
346,347,Это интересно.,https://t.me/s/proeconomics
545,546,Новосибирск.,https://www.militarynews.ru


In [67]:
news["event_title_text"] = (
    news[TITLE_COLUMN]
    .fillna("")
    .astype(str)
)

news["event_body_text"] = (
    news[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.slice(0, 1800)
)


# Если текст почему-то пустой, используем заголовок.
empty_body_mask = (
    news["event_body_text"]
    .str.strip()
    .eq("")
)

news.loc[
    empty_body_mask,
    "event_body_text",
] = news.loc[
    empty_body_mask,
    "event_title_text",
]

In [68]:
title_event_embeddings = embedding_model.encode(
    news["event_title_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

body_event_embeddings = embedding_model.encode(
    news["event_body_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)


title_event_embeddings = np.asarray(
    title_event_embeddings,
    dtype=np.float32,
)

body_event_embeddings = np.asarray(
    body_event_embeddings,
    dtype=np.float32,
)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 59/59 [00:43<00:00,  1.35it/s]


In [69]:
semantic_title_similarity = (
    title_event_embeddings
    @ title_event_embeddings.T
)

semantic_body_similarity = (
    body_event_embeddings
    @ body_event_embeddings.T
)

In [70]:
title_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    max_features=50_000,
)

title_tfidf = title_vectorizer.fit_transform(
    news["event_title_text"]
)

lexical_title_similarity = (
    title_tfidf
    @ title_tfidf.T
).toarray().astype(np.float32)

In [71]:
weak_title_values = (
    news["weak_event_title"]
    .to_numpy(dtype=bool)
)

weak_title_pair = (
    weak_title_values[:, None]
    | weak_title_values[None, :]
)

informative_title_pair = ~weak_title_pair

In [72]:
event_similarity_informative = (
    0.70 * semantic_body_similarity
    + 0.20 * semantic_title_similarity
    + 0.10 * lexical_title_similarity
)

event_similarity_weak = (
    0.95 * semantic_body_similarity
    + 0.05 * semantic_title_similarity
)

event_similarity = np.where(
    informative_title_pair,
    event_similarity_informative,
    event_similarity_weak,
).astype(np.float32)

In [73]:
normalized_titles = (
    news["event_title_normalized"]
    .to_numpy()
)

same_normalized_title = (
    normalized_titles[:, None]
    == normalized_titles[None, :]
)

nonempty_titles = (
    normalized_titles != ""
)

same_nonempty_title = (
    same_normalized_title
    & nonempty_titles[:, None]
    & nonempty_titles[None, :]
)

same_informative_title = (
    same_nonempty_title
    & informative_title_pair
)

In [74]:
event_similarity[
    same_informative_title
] += 0.08

In [75]:
MIN_BODY_SIMILARITY_FOR_WEAK_TITLE = 0.84

weak_title_body_mismatch = (
    weak_title_pair
    & (
        semantic_body_similarity
        < MIN_BODY_SIMILARITY_FOR_WEAK_TITLE
    )
)

In [76]:
event_similarity[
    weak_title_body_mismatch
] = np.minimum(
    event_similarity[
        weak_title_body_mismatch
    ],
    0.79,
)

In [77]:
industry_values = (
    news["industry"]
    .fillna("")
    .astype(str)
    .to_numpy()
)

same_industry = (
    industry_values[:, None]
    == industry_values[None, :]
)

different_industry = ~same_industry

In [78]:
event_similarity[
    different_industry
] -= 0.06

In [79]:
company_values = (
    news[COMPANY_COLUMN]
    .fillna("")
    .astype(str)
    .to_numpy()
)

company_present = (
    company_values != ""
)

both_have_company = (
    company_present[:, None]
    & company_present[None, :]
)

same_company = (
    company_values[:, None]
    == company_values[None, :]
)

same_nonempty_company = (
    both_have_company
    & same_company
)

different_nonempty_company = (
    both_have_company
    & ~same_company
)

In [80]:
event_similarity[
    same_nonempty_company
] += 0.025

event_similarity[
    different_nonempty_company
] -= 0.035

In [81]:
def extract_event_numbers(text: str) -> set[str]:
    text = str(text or "")
    text = text.replace(",", ".")

    numbers = re.findall(
        r"(?<!\w)\d+(?:\.\d+)?%?",
        text,
    )

    return set(numbers)


news["event_numbers"] = (
    news["event_title_text"]
    + " "
    + news["event_body_text"].str.slice(0, 500)
).map(extract_event_numbers)

In [82]:
number_sets = news["event_numbers"].tolist()

In [83]:
for i in range(len(news)):
    numbers_i = number_sets[i]

    if not numbers_i:
        continue

    for j in range(i + 1, len(news)):
        numbers_j = number_sets[j]

        if not numbers_j:
            continue

        if numbers_i.isdisjoint(numbers_j):
            event_similarity[i, j] -= 0.04
            event_similarity[j, i] -= 0.04

In [84]:
event_similarity = np.clip(
    event_similarity,
    0.0,
    1.0,
)

np.fill_diagonal(
    event_similarity,
    1.0,
)

event_distance = (
    1.0 - event_similarity
)

np.fill_diagonal(
    event_distance,
    0.0,
)

In [85]:
EVENT_DISTANCE_THRESHOLD = 0.18

try:
    event_clusterer = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=EVENT_DISTANCE_THRESHOLD,
        compute_full_tree=True,
    )

except TypeError:
    event_clusterer = AgglomerativeClustering(
        n_clusters=None,
        affinity="precomputed",
        linkage="average",
        distance_threshold=EVENT_DISTANCE_THRESHOLD,
        compute_full_tree=True,
    )


raw_event_labels = event_clusterer.fit_predict(
    event_distance
)

In [86]:
event_first_positions = {}

for position, label in enumerate(raw_event_labels):
    if label not in event_first_positions:
        event_first_positions[label] = position


ordered_labels = sorted(
    event_first_positions,
    key=event_first_positions.get,
)

event_label_mapping = {
    label: f"E{index:04d}"
    for index, label in enumerate(
        ordered_labels,
        start=1,
    )
}

news["event_id"] = [
    event_label_mapping[label]
    for label in raw_event_labels
]

print(
    "Количество найденных событий:",
    news["event_id"].nunique(),
)

Количество найденных событий: 1672


In [87]:
# Индекс должен соответствовать позициям строк в embedding-матрицах.
news = news.reset_index(drop=True)

TITLE_COLUMN = (
    "title_clean"
    if "title_clean" in news.columns
    else "title"
)

TEXT_COLUMN = (
    "text_clean"
    if "text_clean" in news.columns
    else "text"
)

SOURCE_COLUMN = (
    "source_clean"
    if "source_clean" in news.columns
    else "source"
)

if "target_client" in news.columns:
    COMPANY_COLUMN = "target_client"
elif "company" in news.columns:
    COMPANY_COLUMN = "company"
else:
    raise ValueError(
        "Не найдена колонка с компанией: "
        "target_client или company."
    )

required_columns = [
    TITLE_COLUMN,
    TEXT_COLUMN,
    SOURCE_COLUMN,
    COMPANY_COLUMN,
    "industry",
    "event_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in news.columns
]

if missing_columns:
    raise ValueError(
        f"Отсутствуют колонки: {missing_columns}"
    )

print("Используемые колонки:")
print("title:", TITLE_COLUMN)
print("text:", TEXT_COLUMN)
print("source:", SOURCE_COLUMN)
print("company:", COMPANY_COLUMN)

Используемые колонки:
title: title_clean
text: text_clean
source: source_clean
company: company


In [88]:
import re
import json
import numpy as np
import pandas as pd
from urllib.parse import urlparse, unquote
import re

In [89]:
SOURCE_ALIASES = {
    # При необходимости сюда можно добавлять ручные объединения:
    # "telegram:banksta": "Telegram: Банкста",
    # "telegram:rbc_news": "Telegram: РБК",
}


def normalize_source_name(source: str) -> str:
    """
    Нормализует источник новости.

    Telegram:
        https://t.me/s/channel_name
        https://t.me/channel_name
        https://t.me/s/channel_name/123
        -> telegram:channel_name

    Обычные сайты:
        https://www.rbc.ru/...
        -> rbc.ru

    Простые текстовые названия:
        РИА Новости
        -> риа новости
    """

    source = str(source or "").strip()

    if not source:
        return "unknown"

    source = unquote(source)
    source_lower = source.lower().strip()

    # Добавляем схему, если перед нами адрес без https://.
    possible_domains = (
        "t.me/",
        "telegram.me/",
        "www.",
    )

    if (
        "://" not in source_lower
        and source_lower.startswith(possible_domains)
    ):
        source_lower = "https://" + source_lower

    # =========================
    # Обработка URL
    # =========================

    if "://" in source_lower:
        try:
            parsed = urlparse(source_lower)

            domain = parsed.netloc.lower()
            domain = re.sub(r"^www\.", "", domain)

            path_parts = [
                part.strip()
                for part in parsed.path.split("/")
                if part.strip()
            ]

            # Telegram-ссылки.
            if domain in {
                "t.me",
                "telegram.me",
                "telegram.dog",
            }:
                # Возможные варианты:
                # /s/channel
                # /s/channel/123
                # /channel
                # /channel/123

                if (
                    len(path_parts) >= 2
                    and path_parts[0].lower() == "s"
                ):
                    channel_name = path_parts[1]

                elif len(path_parts) >= 1:
                    channel_name = path_parts[0]

                else:
                    return "telegram:unknown"

                channel_name = channel_name.lower()
                channel_name = channel_name.lstrip("@")

                # Служебные Telegram-ссылки не являются каналами.
                if channel_name in {
                    "share",
                    "joinchat",
                    "addstickers",
                    "proxy",
                    "socks",
                    "login",
                }:
                    return f"telegram:{channel_name}"

                normalized = f"telegram:{channel_name}"

                return SOURCE_ALIASES.get(
                    normalized,
                    normalized,
                )

            # Для обычного сайта источником считаем домен.
            if domain:
                return SOURCE_ALIASES.get(
                    domain,
                    domain,
                )

        except Exception:
            # Если URL оказался нестандартным,
            # продолжаем как с обычной строкой.
            pass

    # =========================
    # Текстовое название
    # =========================

    normalized = source_lower.replace("ё", "е")

    normalized = re.sub(
        r"[^a-zа-я0-9_.:@-]+",
        " ",
        normalized,
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    return SOURCE_ALIASES.get(
        normalized,
        normalized,
    )

In [90]:
test_sources = [
    "https://t.me/s/banksta",
    "https://t.me/s/banksta/12345",
    "https://t.me/banksta",
    "t.me/s/rbc_news",
    "https://telegram.me/s/rian_ru",
    "https://www.rbc.ru/economics/20/07/2026/article",
    "РИА Новости",
]

for value in test_sources:
    print(
        f"{value:65} -> {normalize_source_name(value)}"
    )

https://t.me/s/banksta                                            -> telegram:banksta
https://t.me/s/banksta/12345                                      -> telegram:banksta
https://t.me/banksta                                              -> telegram:banksta
t.me/s/rbc_news                                                   -> telegram:rbc_news
https://telegram.me/s/rian_ru                                     -> telegram:rian_ru
https://www.rbc.ru/economics/20/07/2026/article                   -> rbc.ru
РИА Новости                                                       -> риа новости


In [91]:
news["source_normalized"] = (
    news[SOURCE_COLUMN]
    .map(normalize_source_name)
)

In [92]:
source_statistics = (
    news["source_normalized"]
    .value_counts()
    .rename_axis("source_normalized")
    .reset_index(name="news_count")
)

display(source_statistics.head(50))

,source_normalized,news_count
0,news.mail.ru,147
1,rg.ru,64
2,ria.ru,62
3,telegram:markettwits,60
4,m24.ru,58
5,life.ru,57
6,osnmedia.ru,56
7,vm.ru,52
8,vedomosti.ru,44
9,1prime.ru,42


In [93]:
display(
    news[
        [
            SOURCE_COLUMN,
            "source_normalized",
        ]
    ]
    .drop_duplicates()
    .sort_values("source_normalized")
    .head(100)
)

,source_clean,source_normalized
563,https://1prime.ru/finance/,1prime.ru
432,https://1prime.ru/state_regulation/,1prime.ru
394,https://1prime.ru/news/,1prime.ru
708,https://academia.interfax.ru/ru/news/,academia.interfax.ru
884,http://adigea.aif.ru/rss/all.php,adigea.aif.ru
...,...,...
578,http://nsn.fm/rss/news.xml,nsn.fm
423,http://nsn.fm/rss/news,nsn.fm
498,https://www.ntv.ru/novosti/,ntv.ru
822,https://www.ntv.ru/economics,ntv.ru


In [94]:
from pathlib import Path
from urllib.parse import urlparse, unquote
import json
import re

import numpy as np
import pandas as pd


# ============================================================
# 1. Базовая подготовка
# ============================================================

news = news.loc[:, ~news.columns.duplicated()].copy()
news = news.reset_index(drop=True)

if "event_id" not in news.columns:
    raise ValueError("В news нет колонки event_id. Сначала выполни кластеризацию событий.")

TITLE_COLUMN = "title_clean" if "title_clean" in news.columns else "title"
TEXT_COLUMN = "text_clean" if "text_clean" in news.columns else "text"
SOURCE_COLUMN = "source_clean" if "source_clean" in news.columns else "source"

# Для компаний из clients.json предпочтительно использовать target_client.
if "target_client" in news.columns:
    COMPANY_COLUMN = "target_client"
elif "company" in news.columns:
    COMPANY_COLUMN = "company"
else:
    COMPANY_COLUMN = "_company"
    news[COMPANY_COLUMN] = None

if "industry" not in news.columns:
    raise ValueError("В news нет колонки industry.")

if "news_id" not in news.columns:
    news["news_id"] = np.arange(1, len(news) + 1)


# ============================================================
# 2. Правильная нормализация Telegram-источников
# ============================================================

def normalize_source_name(source):
    source = str(source or "").strip()

    if not source:
        return "unknown"

    source = unquote(source)
    source_lower = source.lower().strip()

    # Добавляем протокол для адресов вида t.me/s/channel
    if "://" not in source_lower and source_lower.startswith(
        ("t.me/", "telegram.me/", "telegram.dog/", "www.")
    ):
        source_lower = "https://" + source_lower

    if "://" in source_lower:
        parsed = urlparse(source_lower)

        domain = parsed.netloc.lower()
        domain = re.sub(r"^www\.", "", domain)

        path_parts = [
            part
            for part in parsed.path.split("/")
            if part
        ]

        # Telegram
        if domain in {"t.me", "telegram.me", "telegram.dog"}:
            # https://t.me/s/channel_name/123
            if len(path_parts) >= 2 and path_parts[0].lower() == "s":
                channel_name = path_parts[1]

            # https://t.me/channel_name/123
            elif len(path_parts) >= 1:
                channel_name = path_parts[0]

            else:
                return "telegram:unknown"

            channel_name = channel_name.lower().lstrip("@")

            return f"telegram:{channel_name}"

        # Обычный сайт
        if domain:
            return domain

    # Источник задан обычным текстом
    source_lower = source_lower.replace("ё", "е")

    source_lower = re.sub(
        r"[^a-zа-я0-9_.:@-]+",
        " ",
        source_lower,
    )

    source_lower = re.sub(
        r"\s+",
        " ",
        source_lower,
    ).strip()

    return source_lower or "unknown"


# Пересчитываем всегда, даже если колонка уже существовала.
news["source_normalized"] = (
    news[SOURCE_COLUMN]
    .map(normalize_source_name)
)


# ============================================================
# 3. Удаляем старые результаты цитируемости
# ============================================================

old_columns = [
    "event_representative_title",
    "event_articles_count",
    "event_sources_count",
    "event_sources",
    "event_industry",
    "event_company",
    "citation_count",
    "citation_index",
    "citation_level",
]

news = news.drop(
    columns=[
        column
        for column in old_columns
        if column in news.columns
    ],
    errors="ignore",
)


# ============================================================
# 4. Вспомогательные функции
# ============================================================

def most_common_nonempty(values):
    values = pd.Series(values).dropna().astype(str)
    values = values[values.str.strip() != ""]

    if len(values) == 0:
        return None

    return values.value_counts().index[0]


def representative_title(values):
    """
    Берём наиболее длинный непустой заголовок кластера.
    Поэтому вместо слабого заголовка 'Москва' по возможности
    будет выбран более содержательный заголовок.
    """
    values = pd.Series(values).dropna().astype(str)
    values = values[values.str.strip() != ""]

    if len(values) == 0:
        return ""

    return max(values.tolist(), key=len)


def unique_sources(values):
    result = {
        str(value).strip()
        for value in values
        if pd.notna(value)
        and str(value).strip()
        and str(value).strip() not in {
            "unknown",
            "telegram:unknown",
        }
    }

    return sorted(result)


# ============================================================
# 5. Статистика для каждого события
# ============================================================

event_statistics = (
    news.groupby(
        "event_id",
        sort=False,
    )
    .agg(
        event_articles_count=(
            "news_id",
            "size",
        ),
        event_representative_title=(
            TITLE_COLUMN,
            representative_title,
        ),
        event_sources=(
            "source_normalized",
            unique_sources,
        ),
        event_industry=(
            "industry",
            most_common_nonempty,
        ),
        event_company=(
            COMPANY_COLUMN,
            most_common_nonempty,
        ),
    )
    .reset_index()
)


event_statistics["event_sources_count"] = (
    event_statistics["event_sources"]
    .map(len)
)


# ============================================================
# 6. Цитируемость
# ============================================================

# Один источник — исходная публикация.
# Остальные источники считаются растиражировавшими событие.
event_statistics["citation_count"] = (
    event_statistics["event_sources_count"] - 1
).clip(lower=0)


max_citation_count = (
    event_statistics["citation_count"].max()
)

if max_citation_count > 0:
    event_statistics["citation_index"] = (
        100
        * np.log1p(event_statistics["citation_count"])
        / np.log1p(max_citation_count)
    )
else:
    event_statistics["citation_index"] = 0.0

event_statistics["citation_index"] = (
    event_statistics["citation_index"]
    .round(2)
)


def get_citation_level(source_count):
    if source_count <= 1:
        return "не растиражирована"
    elif source_count <= 3:
        return "низкая"
    elif source_count <= 7:
        return "средняя"
    else:
        return "высокая"


event_statistics["citation_level"] = (
    event_statistics["event_sources_count"]
    .map(get_citation_level)
)


# ============================================================
# 7. Добавляем результаты к каждой новости
# ============================================================

news = news.merge(
    event_statistics,
    on="event_id",
    how="left",
    validate="many_to_one",
)


# ============================================================
# 8. Формируем финальную таблицу
# ============================================================

final_data = {
    "news_id": news["news_id"],
    "title": news[TITLE_COLUMN],
    "text": news[TEXT_COLUMN],
    "source": news[SOURCE_COLUMN],
    "source_normalized": news["source_normalized"],
    "company": news[COMPANY_COLUMN],
    "industry": news["industry"],
    "event_id": news["event_id"],
    "event_representative_title": news["event_representative_title"],
    "event_articles_count": news["event_articles_count"],
    "event_sources_count": news["event_sources_count"],
    "citation_count": news["citation_count"],
    "citation_index": news["citation_index"],
    "citation_level": news["citation_level"],
}

# Добавляем оценки отрасли, только если они существуют.
if "industry_score" in news.columns:
    final_data["industry_score"] = news["industry_score"]

if "industry_confidence" in news.columns:
    final_data["industry_confidence"] = news["industry_confidence"]


final_news = pd.DataFrame(final_data)


# ============================================================
# 9. Сохраняем файлы
# ============================================================

OUTPUT_DIR = Path(r"D:\sber\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_NEWS_PATH = OUTPUT_DIR / "news_final.csv"
EVENTS_PATH = OUTPUT_DIR / "events.csv"


final_news.to_csv(
    FINAL_NEWS_PATH,
    index=False,
    encoding="utf-8-sig",
)


events_output = event_statistics.copy()

events_output["event_sources"] = (
    events_output["event_sources"]
    .map(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
        )
    )
)

events_output.to_csv(
    EVENTS_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 10. Результат
# ============================================================

print("Готово.")
print("Количество новостей:", len(final_news))
print("Количество событий:", final_news["event_id"].nunique())
print("Максимальное число источников:", final_news["event_sources_count"].max())
print()
print("Финальный файл:")
print(FINAL_NEWS_PATH)
print()
print("Таблица событий:")
print(EVENTS_PATH)

display(final_news.head(10))

Готово.
Количество новостей: 1880
Количество событий: 1672
Максимальное число источников: 9

Финальный файл:
D:\sber\outputs\news_final.csv

Таблица событий:
D:\sber\outputs\events.csv


,news_id,title,text,source,source_normalized,company,industry,event_id,event_representative_title,event_articles_count,event_sources_count,citation_count,citation_index,citation_level,industry_score,industry_confidence
0,1,Начался пожар после взрыва на крупнейшем в мир...,Начался пожар после взрыва на крупнейшем в мир...,https://t.me/s/banksta,telegram:banksta,None,нефтегаз,E0001,Начался пожар после взрыва на крупнейшем в мир...,1,1,0,0.00,не растиражирована,0.522087,high
1,2,Отдых россиян в Азии снизил инфляцию в России.,Отдых россиян в Азии снизил инфляцию в России....,https://t.me/s/bankrollo,telegram:bankrollo,None,экономика,E0002,Отдых россиян в Азии снизил инфляцию в России.,1,1,0,0.00,не растиражирована,0.527688,high
2,3,Правительство подготовило новую версию законоп...,Правительство подготовило новую версию законоп...,https://t.me/s/rbc_news,telegram:rbc_news,None,политика,E0003,Правительство подготовило новую версию законоп...,1,1,0,0.00,не растиражирована,0.338848,high
3,4,✨ Закрыт сбор на мобильные огневые группы!,✨ Закрыт сбор на мобильные огневые группы! ✨ З...,https://t.me/s/dva_majors,telegram:dva_majors,None,телекоммуникации,E0004,✨ Закрыт сбор на мобильные огневые группы!,1,1,0,0.00,не растиражирована,0.287161,low
4,5,"↖️ Индия, один из ключевых покупателей российс...","↖️ Индия, один из ключевых покупателей российс...",https://t.me/s/kommersant,telegram:kommersant,None,нефтегаз,E0005,"↖️ Индия, один из ключевых покупателей российс...",2,2,1,31.55,низкая,0.540161,high
5,6,Wildberries & Russ отложила планы по развитию ...,Wildberries & Russ отложила планы по развитию ...,https://t.me/s/kommersant,telegram:kommersant,None,потребительский сектор,E0006,🤪 Wildberries заморозила развитие игрового нап...,2,2,1,31.55,низкая,0.097967,low
6,7,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,https://t.me/s/kommersant,telegram:kommersant,None,экономика,E0007,🗞🗞🗞🗞 По итогам пяти месяцев текущего года Альф...,1,1,0,0.00,не растиражирована,0.323296,high
7,8,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026. 📬 mt ...,https://t.me/s/markettwits,telegram:markettwits,None,экономика,E0008,📬 mt в max 🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.,1,1,0,0.00,не растиражирована,0.367123,high
8,9,😀 РынкиДеньгиВласть — это аналитический сервис...,😀 РынкиДеньгиВласть — это аналитический сервис...,https://t.me/s/AK47pfl,telegram:ak47pfl,None,потребительский сектор,E0009,😀 РынкиДеньгиВласть — это аналитический сервис...,1,1,0,0.00,не растиражирована,0.457881,low
9,10,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,https://t.me/s/markettwits,telegram:markettwits,None,экономика,E0010,❗️🇨🇳#дкп #китай #цбКНР - 1 YEAR LPR (LOAN PRIM...,1,1,0,0.00,не растиражирована,0.388662,high


In [95]:
most_cited_events = (
    event_statistics
    .sort_values(
        [
            "citation_index",
            "event_sources_count",
            "event_articles_count",
        ],
        ascending=False,
    )
)

display(
    most_cited_events.head(30)
)

,event_id,event_articles_count,event_representative_title,event_sources,event_industry,event_company,event_sources_count,citation_count,citation_index,citation_level
383,E0384,10,"В России средний размер пенсии, превышающей 30...","[kamchatka.aif.ru, m24.ru, news.mail.ru, nsn.f...",экономика,None,9,8,100.00,высокая
419,E0420,10,Четырнадцать новых методов лечения вошли в баз...,"[kolomna-spravka.ru, m24.ru, news.mail.ru, osn...",наука,None,9,8,100.00,высокая
957,E0958,7,"Самолёт авиакомпании «Победы», следовавший из ...","[aviaport.ru, life.ru, m24.ru, ria.ru, russian...",транспорт,None,7,6,88.56,средняя
1291,E1292,7,Investing.com -- Взрыв на заводе по переработк...,"[bel.aif.ru, life.ru, m24.ru, osnmedia.ru, ru....",нефтегаз,None,7,6,88.56,средняя
138,E0139,6,Федеральная антимонопольная служба (ФАС) РФ и ...,"[1prime.ru, akm.ru, energyland.info, pnp.ru, t...",нефтегаз,None,6,5,81.55,средняя
1056,E1057,6,"Самолет, вынужденно приземлившийся в аэропорту...","[aviaport.ru, krym.aif.ru, nsn.fm, rg.ru, ria....",транспорт,None,6,5,81.55,средняя
24,E0025,6,"Более 50 человек пострадали, 18 пропали без ве...","[kolomna-spravka.ru, news.mail.ru, ria.ru, tel...",промышленность,None,5,4,73.25,средняя
183,E0184,4,Американский экономист и многолетний глава Фед...,"[telegram:banksta, telegram:bbbreaking, telegr...",экономика,None,4,3,63.09,средняя
233,E0234,4,⛽️ Акции «Газпрома» впервые с 2009 года упали ...,"[telegram:headlines_for_traders, telegram:komm...",экономика,None,4,3,63.09,средняя
524,E0525,4,Магнитно-резонансную томографию (МРТ) можно бу...,"[m24.ru, russian.rt.com, vedomosti.ru, vm.ru]",наука,None,4,3,63.09,средняя
